In [1]:
from dbrepo.RestClient import RestClient
from requests.auth import HTTPBasicAuth
import requests

BASE_URL = "https://test.dbrepo.tuwien.ac.at/api/v1"

database_id = "b23492bd-f66d-4663-a1f5-f296767dbbdc"

username = "52308278"
password = "#eZOUqcYqe8iZlkIYVzcq#"

auth = HTTPBasicAuth(username, password)

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username=username,
    password=password
)

In [2]:
# --------------------------------------------------
# Get tables
# --------------------------------------------------

tables = client.get_tables(database_id=database_id)

for table in tables:
    print(table.id, table.name)

9365361e-d7f8-42d3-9c8b-543cc8477e99 accident_casualty
fccb4bde-6dcb-4e63-9358-945d7b1c93fc accident
872ede6c-f23d-4129-a179-3094b4a596fe person
af0c82b5-29d1-4376-b4dc-587a7daf8d6b sex
5f8d9948-b3de-4c9a-8cce-55adb93750fb casualty_severity
f6df0087-e822-43e0-b784-86a05bd61db6 casualty_class
720ad88b-6d0a-4208-9349-fc30f4640896 road_class
627bd0c6-c7d2-44a8-93ce-6e9004762eb0 road_surface
0cae0220-4b17-414e-b163-8113cc89b900 vehicle
66486c7c-6659-4083-9489-c018ed0d8900 lighting
ae9236cd-437f-45a7-a49f-75a58984e277 weather


In [3]:
# --------------------------------------------------
# Show columns
# --------------------------------------------------

for table in tables:

    print(f"\nTABLE: {table.name}")

    details = client.get_table(
        database_id=database_id,
        table_id=table.id
    )

    for col in details.columns:
        print(col.id, col.name)


TABLE: accident_casualty
7a62a2e0-72a0-474c-bab3-c301d94fb5f0 reference_number
bf8af9af-df7e-4af9-ae2b-8641cc849f9a person_id
4d272416-6267-4e0b-9a33-58d95c8eeb40 casualty_class_id
fa47d1a8-42b2-4270-ab20-5431711d4e42 casualty_severity_id
232c31c2-6fe7-4085-b414-122cc9164f92 vehicle_id

TABLE: accident
cef1e897-0646-45ad-8275-c7b624ef95de reference_number
9e3bbaa1-3ab8-4850-beb9-3ba55b2305ee easting
85573253-6384-4e5b-b8aa-963b6dc75625 northing
57977d2b-29fa-4a63-a8cb-fb605846fcad number_of_vehicles
351e4210-b9f5-4eef-a7b1-3e4edaefca96 date
4bc272b4-0add-459a-bd45-992d360d6483 time
87b64d39-d11c-4a4a-8c28-800793527784 road_class_id
f0b21c39-1619-4d47-a918-ba634ca7c2c0 road_surface_id
924e0223-f0ee-4494-8758-2c46fa483347 lighting_id
aca6641a-9189-459a-8652-24c7d7d8ec8e weather_id

TABLE: person
d9ae09b3-e796-4ac5-b29e-2912924562a4 person_id
71667cbe-b821-424c-9182-d5d5a78f4ed9 sex_id
7039995b-4e8e-4c9e-b54e-f4785544a890 age

TABLE: sex
e2f08c59-3c57-40cf-989e-7c2ada061ca6 sex_id
1c0470

In [4]:
# --------------------------------------------------
# Ontology Mapping
# --------------------------------------------------

ontology_mapping = {

    ("weather", "description"):
        "http://sweetontology.net/realmAtmoWeather",

    ("lighting", "description"):
        "http://sweetontology.net/phenAtmoLightning",

    ("vehicle", "description"):
        "https://dbpedia.org/ontology/Automobile",

    ("road_surface", "description"):
        "http://purl.obolibrary.org/obo/ENVO_00000064",

    ("road_class", "description"):
        "http://purl.obolibrary.org/obo/ENVO_00000064",

    ("casualty_class", "description"):
        "http://purl.obolibrary.org/obo/NCIT_C212267",

    ("casualty_severity", "description"):
        "http://snomed.info/id/272141005",

    ("sex", "description"):
        "http://snomed.info/id/734000001",

    ("accident", "reference_number"):
        "http://purl.org/dc/terms/identifier",

    ("accident", "easting"):
        "http://www.opengis.net/ont/geosparql#SpatialObject",

    ("accident", "northing"):
        "http://www.opengis.net/ont/geosparql#SpatialObject",

    ("accident", "number_of_vehicles"):
        "http://qudt.org/schema/qudt/Quantity",

    ("accident", "date"):
        "http://www.w3.org/2006/time#Instant",

    ("accident", "time"):
        "http://www.w3.org/2006/time#Instant",

    ("person", "age"):
        "http://purl.obolibrary.org/obo/NCIT_C25150",
}

In [7]:
# --------------------------------------------------
# Update DBRepo Concept URIs
# --------------------------------------------------

for table in tables:

    table_name = table.name
    table_id = table.id

    print(f"\nProcessing {table_name}")

    details = client.get_table(
        database_id=database_id,
        table_id=table_id
    )

    for col in details.columns:

        column_name = col.name
        column_id = col.id

        key = (table_name, column_name)

        if key not in ontology_mapping:
            continue
        
        concept_uri = ontology_mapping[key]

        update_url = (
            f"{BASE_URL}/database/{database_id}"
            f"/table/{table_id}"
            f"/column/{column_id}"
        )

        payload = {
            "concept_uri": concept_uri
        }

        response = requests.put(
            update_url,
            json=payload,
            auth=auth
        )

        print(
            f"{table_name}.{column_name} "
            f"-> {concept_uri} "
            f"(Status: {response.status_code})"
        )

        if response.status_code != 200:
            print(response.text)


Processing accident_casualty

Processing accident
accident.reference_number -> http://purl.org/dc/terms/identifier (Status: 202)

accident.easting -> http://www.opengis.net/ont/geosparql#SpatialObject (Status: 202)

accident.northing -> http://www.opengis.net/ont/geosparql#SpatialObject (Status: 202)

accident.number_of_vehicles -> http://qudt.org/schema/qudt/Quantity (Status: 202)

accident.date -> http://www.w3.org/2006/time#Instant (Status: 202)

accident.time -> http://www.w3.org/2006/time#Instant (Status: 202)


Processing person
person.age -> http://purl.obolibrary.org/obo/NCIT_C25150 (Status: 202)


Processing sex
sex.description -> http://snomed.info/id/734000001 (Status: 202)


Processing casualty_severity
casualty_severity.description -> http://snomed.info/id/272141005 (Status: 202)


Processing casualty_class
casualty_class.description -> http://purl.obolibrary.org/obo/NCIT_C212267 (Status: 202)


Processing road_class
road_class.description -> http://purl.obolibrary.org/ob